In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

In [0]:

storage_account = "projecttraffic60302085"
storage_account_key = ""
spark.conf.set(
 f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
 ""
)



# -------------------------
# 1. Paths
# -------------------------
processed_path = f"abfss://processed@{storage_account}.dfs.core.windows.net/accidents/"
curated_path = f"abfss://curated@{storage_account}.dfs.core.windows.net/accidents_features_v1/"

In [0]:
# -------------------------
# 2. Load Processed Data
# -------------------------
df = spark.read.parquet(processed_path)

print("Initial rows:", df.count())
print("Initial columns:", len(df.columns))
df.printSchema()
display(df.limit(5))







Initial rows: 7728394
Initial columns: 51
root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: string (nullable = true)
 |-- Start_Time: string (nullable = true)
 |-- End_Time: string (nullable = true)
 |-- Start_Lat: string (nullable = true)
 |-- Start_Lng: string (nullable = true)
 |-- End_Lat: string (nullable = true)
 |-- End_Lng: string (nullable = true)
 |-- Distancemi: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: string (nullable = true)
 |-- TemperatureF: string (nullable = true)
 |-- Wind_ChillF: string (nullable = true)
 |-- Humidity: string (nullable = true)
 |-- Pressurein: string (nu

ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distancemi,Description,Street,City,County,State,Zipcode,Country,Timezone,Airport_Code,Weather_Timestamp,TemperatureF,Wind_ChillF,Humidity,Pressurein,Visibilitymi,Wind_Direction,Wind_Speedmph,Precipitationin,Weather_Condition,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight,HourOfDay,Month,IsRushHour,WeatherSeverityMapped,AccidentYear
A-5829313,Source1,2,2021-03-09 15:33:30,2021-03-09 16:51:30,25.913224,-80.336809,25.913248,-80.335034,0.11,Incident on NW 154TH ST near NW 84TH BLK Drive with caution.,NW 154th St,Hialeah,Miami-Dade,FL,33016,US,US/Eastern,KOPF,2021-03-09 15:53:00,71.0,71.0,49.0,30.34,10.0,ENE,18.0,0.0,Mostly Cloudy,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day,15,3,0,4,2021
A-5829314,Source1,2,2021-12-09 13:22:00,2021-12-09 15:23:55,36.123483,-86.849664,36.124698,-86.847346,0.154,Stationary traffic on US-70S E - US-70S W - TN-100 - TN-1 - TN-155 from Hillwood Blvd / Lynnwood Blvd to Ashley Park Dr due to accident.,Harding Pike,Nashville,Davidson,TN,37205-2102,US,US/Central,KBNA,2021-12-09 12:53:00,62.0,62.0,32.0,29.32,10.0,SSW,15.0,0.0,Partly Cloudy,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day,13,12,0,4,2021
A-5829315,Source1,2,2021-11-10 15:21:00,2021-11-10 18:03:18,38.548803,-121.509908,38.561157,-121.514353,0.887,Stationary traffic on I-5 N from exit [516] to exit [518] due to accident.,I-5 N,Sacramento,Sacramento,CA,95818,US,US/Pacific,KSAC,2021-11-10 14:53:00,63.0,63.0,78.0,30.26,10.0,NNW,7.0,0.0,Fair,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day,15,11,0,1,2021
A-5829316,Source1,2,2021-09-19 11:49:16,2021-09-19 13:16:18,37.83615231495935,-79.36981558761899,37.80469231495935,-79.39098558761899,2.462,Incident on I-81 SB near I-81 Drive with caution.,I-64,Lexington,Rockbridge,VA,24450,US,US/Eastern,KHSP,2021-09-19 11:55:00,73.0,73.0,63.0,26.42,10.0,E,6.0,0.0,Fair,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day,11,9,0,1,2021
A-5829317,Source1,2,2021-08-19 13:21:02,2021-08-19 17:02:19,25.93785,-80.27734699999998,25.939753,-80.278228,0.142,Slow traffic on FL-847 - FL-860 from NW 181st St to NW 48th Pl due to accident.,NW 181st St,Opa Locka,Miami-Dade,FL,33055,US,US/Eastern,KOPF,2021-08-19 12:53:00,90.0,90.0,57.0,30.08,10.0,ENE,8.0,0.0,Fair,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day,13,8,0,1,2021


In [0]:
# -------------------------
# 3. Type Conversions
# -------------------------

# Convert datetime columns if needed
if "Start_Time" in df.columns:
    df = df.withColumn("Start_Time", F.to_timestamp("Start_Time"))

if "End_Time" in df.columns:
    df = df.withColumn("End_Time", F.to_timestamp("End_Time"))

if "Weather_Timestamp" in df.columns:
    df = df.withColumn("Weather_Timestamp", F.to_timestamp("Weather_Timestamp"))

# Convert integer-like columns
int_columns = [
    "Severity",
    "accidentYear",
    "hourOfDay",
    "month",
    "isRushHour",
    "weatherSeverityMapped"
]

for c in int_columns:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("int"))

# Optional: convert coordinates to double if needed
double_columns = ["Start_Lat", "Start_Lng", "End_Lat", "End_Lng"]
for c in double_columns:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("double"))

In [0]:
# -------------------------
# 4. Basic Cleaning
# -------------------------

# Drop exact duplicates
df = df.dropDuplicates()

# Handle missing values in key columns
df = df.filter(F.col("Start_Time").isNotNull())
df = df.filter(F.col("Severity").isNotNull())
df = df.filter(F.col("Weather_Condition").isNotNull())

# Optional: fill missing categorical values if needed
df = df.fillna({
    "City": "Unknown"
})

In [0]:
# -------------------------
# 5. Validation Checks
# -------------------------
print("Missing values in selected columns:")
display(
    df.select(
        F.sum(F.col("Start_Time").isNull().cast("int")).alias("missing_Start_Time"),
        F.sum(F.col("Severity").isNull().cast("int")).alias("missing_Severity"),
        F.sum(F.col("Weather_Condition").isNull().cast("int")).alias("missing_Weather_Condition"),
        F.sum(F.col("City").isNull().cast("int")).alias("missing_City")
    )
)

print("Range checks:")
display(
    df.select(
        F.min("hourOfDay").alias("min_hour"),
        F.max("hourOfDay").alias("max_hour"),
        F.min("month").alias("min_month"),
        F.max("month").alias("max_month"),
        F.min("accidentYear").alias("min_year"),
        F.max("accidentYear").alias("max_year")
    )
)

Missing values in selected columns:


missing_Start_Time,missing_Severity,missing_Weather_Condition,missing_City
0,0,0,0


Range checks:


min_hour,max_hour,min_month,max_month,min_year,max_year
0,23,1,12,2016,2023


In [0]:
# -------------------------
# 6. Save Curated Data
# -------------------------
df.write \
  .mode("overwrite") \
  .parquet(curated_path)

print("Cleaned curated dataset saved successfully.")

Cleaned curated dataset saved successfully.
